# Phase 1: evasive answers

Release `data/results/2026-09-23/` (corrected), earnings calls only. Plan: `plans/phase_1.md` with its amendments of 2026-09-23. Computation: `analysis/phase1_score.py`, `analysis/phase1_inference.py`, `analysis/phase1_labelling.py`; outputs in `data/phase1/`.

The earlier release `2026-09-13` had lost every `n't` in its sentence segmentation. As fixed in the plan before the corrected release existed, the primary tests on `2026-09-23` are the primary results, and the run on `2026-09-13` (`data/phase1_faulty_release_2026-09-13/`) is reported next to them as a run on faulty data. **All results are provisional** until the labelling samples are returned and the validation gate is applied.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "roman" else Path.cwd()
DATA = ROOT / "notebooks" / "roman" / "data"
OUT = DATA / "phase1"
OLD = DATA / "phase1_faulty_release_2026-09-13"
COLORS = {"UBS": "#2a78d6", "JPM": "#eb6834"}
MARKERS = {"UBS": "o", "JPM": "s"}
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 3)

## Data and detector

In [ ]:
seg = pd.read_csv(OUT / "segments.csv")
meta = json.loads((OUT / "phase1_meta.json").read_text())
pd.DataFrame({"question turns": seg.groupby("bank").pair_id.nunique(),
              "question segments": seg.groupby("bank").size(),
              "segments on capital regulation (keywords)": seg.groupby("bank").capital_kw.sum(),
              "share of answer sentences under 4 words": pd.Series(meta["short_answer_share"])})

UBS turns split into 2.6 segments on average and JPMorgan turns into 1.7, which matches the bundling of UBS questions. The capital-regulation keywords match 10 % of UBS and 8 % of JPMorgan segments.

In [ ]:
pd.read_csv(OUT / "e3_agreement.csv").drop(columns="restored_phrase")

*Exploratory (E3).* On the corrected release the phrase list flags 17 UBS and 7 JPMorgan sentences, and NLI finds most of them as well; six UBS and two JPMorgan sentences are found by the phrase list alone. NLI adds 21 UBS and 15 JPMorgan sentences that the phrase list misses.

## P1-1: deflection rate, UBS vs JPMorgan (primary)

In [ ]:
rates = pd.read_csv(OUT / "call_rates.csv")
r = rates[rates.variant == "primary"]
fig, ax = plt.subplots(figsize=(8, 4))
for bank, g in r.groupby("bank"):
    g = g.sort_values("quarter")
    ax.plot(g.quarter, g.rate, marker=MARKERS[bank], color=COLORS[bank], lw=2, ms=8, label=bank)
    ax.annotate(bank, (g.quarter.iloc[-2], g.rate.iloc[-2]), xytext=(0, 12 if bank == "UBS" else -18),
                textcoords="offset points", ha="center", color="0.2")
ax.set_ylabel("Deflection sentences per 100\nmanagement answer sentences")
ax.set_xlabel("Reporting quarter of the earnings call")
ax.set_ylim(0, r.rate.max() * 1.2)
ax.grid(axis="y", color="0.9"); ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
ax.set_title("Deflection rate per call, primary detector (phrase list or NLI >= 0.9), release 2026-09-23",
             loc="left", fontsize=10)
plt.tight_layout()

In [ ]:
p1 = pd.read_csv(OUT / "p1_deflection_rate.csv")
p1["role"] = ["primary"] + ["robustness"] * 6
p1

UBS deflects in 3.0 of 100 answer sentences against 1.1 at JPMorgan, a difference of 1.8 with an exact call-level p of 0.040 (12,870 splits). After Holm correction across the three primary tests of the study p is 0.080, so by the pre-registered criterion the difference is not supported.

In [ ]:
same = p1[p1.variant != "per_10k_words"]
pd.DataFrame({"T primary": [p1["T"][0]], "min over variants": [same["T"].min()], "max over variants": [same["T"].max()],
              "variants with T > 0": [f"{(p1['T'] > 0).sum()} of {len(p1)}"],
              "variants with p < 0.05": [f"{(p1.p < 0.05).sum()} of {len(p1)}"]})

The effect ranges from 0.90 to 2.00 deflection sentences per 100 answer sentences over the variants on this scale; the rate per 10,000 words is on a different scale (6.5) and is left out of the range.

The direction holds in every robustness variant, with raw p between 0.023 and 0.062; only the rate per 10,000 words lies above 0.05. The difference is stable in sign but borderline in size, and it rests on eight calls per bank.

## P1-2: concentration of deflections on capital regulation (primary)

In [ ]:
p2 = pd.read_csv(OUT / "p2_capital_concentration.csv")
p2["role"] = ["primary"] + ["robustness"] * 6
p2

In [ ]:
pd.DataFrame({"T primary": [p2["T"][0]], "min over variants": [p2["T"].min()], "max over variants": [p2["T"].max()],
              "variants with T > 0": [f"{(p2['T'] > 0).sum()} of {len(p2)}"],
              "variants with p < 0.05": [f"{(p2.p < 0.05).sum()} of {len(p2)}"]})

The excess share of deflected segments on capital regulation is 0.13 at UBS and 0.03 at JPMorgan, a difference of 0.10 with p = 0.16. The hypothesis is not supported; over the variants T ranges from 0.06 to 0.17 with p between 0.07 and 0.26, all in the same direction.

In [ ]:
segf = pd.read_csv(OUT / "segments_flagged.csv")
segf[(segf.bank == "UBS") & segf.deflected & (segf.quarter >= "2024")][["quarter", "capital", "text"]]

Several deflected UBS segments of 2024 ask about mitigation, foreign subsidiaries or participations after the regime context was cut off by the splitting rule, and are therefore labelled as other. The weak P1-2 result is at least partly a measurement problem of the topic label, which the topic labelling sample will quantify.

## Corrected release against the run on faulty data

In [ ]:
def side_by_side(name, cols):
    new = pd.read_csv(OUT / name).set_index("variant")[cols]
    old = pd.read_csv(OLD / name).set_index("variant")[cols]
    old = old.drop(index=[v for v in old.index if v.startswith("E4")])
    return pd.concat({"2026-09-23 (primary)": new, "2026-09-13 (faulty data)": old}, axis=1)

side_by_side("p1_deflection_rate.csv", ["mean_UBS", "mean_JPM", "T", "p"])

In [ ]:
side_by_side("p2_capital_concentration.csv", ["E_UBS", "E_JPM", "T", "p", "D_UBS", "D_JPM"])

In [ ]:
reg_old = pd.read_csv(OLD / "test_register_before_rerun.csv").set_index("test_id")[["p_raw", "p_holm"]]
reg_new = pd.read_csv(ROOT / "notebooks" / "roman" / "plans" / "test_register.csv").set_index("test_id")[["p_raw", "p_holm"]]
pd.concat({"2026-09-23 (primary)": reg_new, "2026-09-13 (faulty data)": reg_old}, axis=1)

Restoring the contractions changes the P1-1 difference only slightly (1.91 to 1.81) and moves p from 0.029 to 0.040; after Holm neither run reaches 0.05. P1-2 moves from p = 0.39 to 0.16 in the same direction, still far from support.

## E1: timing, Swiss regulatory debate 2024 vs integration 2023 (exploratory)

In [ ]:
pd.read_csv(OUT / "e1_timing.csv")

*Exploratory.* The UBS rate rises by 2.6 points from 2023 to 2024 against 0.7 at JPMorgan, a difference in differences of 1.9 with p = 0.25 (2.2 and p = 0.20 from 2023-Q4). Deflections on keyword-labelled capital questions do not rise at UBS, so the timing hypothesis finds no support in this label.

In [ ]:
pd.read_csv(OUT / "e1_content.csv")

*Exploratory.* Two deflected UBS segments in 2023 and one in 2024 contain Swiss-regime terms, three in 2023 contain integration terms; the keyword content check is too thin to separate the two explanations.

## E2: answer relevance (exploratory)

In [ ]:
rel = pd.read_csv(OUT / "relevance_scored.csv")
(rel.groupby(["model", "answer_variant", "bank"])
    .agg(mean_lift=("lift", "mean"), mean_lift_length_adjusted=("lift_resid", "mean"),
         share_above_null=("lift", lambda x: (x > 0).mean()), segments=("lift", "size")))

In [ ]:
(rel[rel.answer_variant == "first128"].groupby(["model", "bank", "deflected"]).lift.mean()
    .unstack("deflected").rename(columns={False: "not deflected", True: "deflected"}))

*Exploratory.* Both models put the true answer above the other answers of the call for more than 80 % of segments, with no difference between banks. Deflected answers are not less relevant than others, because a deflection usually repeats the question's topic, so relevance scores do not detect evasion; the bi-encoder lift is also partly circular, since the same model aligned the answers.

## E5: topic per whole analyst turn (exploratory)

In [ ]:
pd.read_csv(OUT / "e5_summary.csv")

*Exploratory, formed after the first results.* With the topic taken from the whole analyst turn, 10 of 27 deflected UBS pairs concern capital regulation against 3 of 21 at JPMorgan. Because 22 % of UBS turns but only 11 % of JPMorgan turns mention capital regulation, much of this difference is expected from the question mix alone.

In [ ]:
pd.read_csv(OUT / "e5_concentration.csv")

*Exploratory.* The excess share over the within-call expectation is 0.13 at UBS and 0.03 at JPMorgan, a difference of 0.10 with p = 0.29. The turn-level label is coarse, because bundled UBS turns often mix a capital question with others.

In [ ]:
pd.read_csv(OUT / "e5_timing.csv")

*Exploratory.* Deflections in capital-regulation pairs rise slightly more at UBS than at JPMorgan from 2023 to 2024 (difference in differences 0.34 per 100 sentences, p = 0.64; 0.26 and p = 0.77 from 2023-Q4). With either topic definition the data do not show that UBS deflections follow the Swiss regulatory debate.

## Validation samples

`data/labelling/deflection.csv` (180 sentences: all 60 flagged sentences, 10 near-threshold sentences and 50 unflagged sentences per bank, half of them from answers to capital questions) and `data/labelling/question_topic.csv` (100 segments, 25 per bank and keyword outcome) are drawn from the corrected release and are blind; strata, scores and weights are in the `_key.csv` files. The samples from `2026-09-13` are kept in `data/labelling/superseded_release_2026-09-13/` and are not labelled.